# Video to Speech Pipeline

This notebook implements an end-to-end pipeline to:
1.  **Extract meaningful frames** from a tactical video (`.mp4`).
2.  **Generate a tactical description** using a Vision-Language Model (VLM).
3.  **Synthesize the description into speech** using Neural TTS.

In [1]:
import pandas as pd
import numpy as np
import os
import os.path as osp
from mplsoccer import Pitch
from highlight_text import fig_text
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import matplotlib
import sys

# Video-to-speech
import cv2
import base64
import ollama
import edge_tts
import asyncio
import nest_asyncio
from IPython.display import HTML, Audio, display, Video

# Patch asyncio for Jupyter
nest_asyncio.apply()

print("✅ Libraries imported. Local VLM ready.")

# In notebooks, __file__ is not defined, so use os.getcwd()
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from utils import build_pro_tactical_phases, get_events_by_possession_id

✅ Libraries imported. Local VLM ready.


## Utils

### Parte I: Extracción de Fases de Juego

In [2]:
def load_tracking_as_mplsoccer(home_file, away_file, home_team='Home', away_team='Away'):

    # Cargar los datos
    df_home = pd.read_csv(home_file, skiprows=2)
    df_home.sort_values('Time [s]', inplace=True)

    df_away = pd.read_csv(away_file, skiprows=2)
    df_away.sort_values('Time [s]', inplace=True)

    for df in [df_home, df_away]:
        cols = list(np.repeat(df.columns[3::2], 2))
        cols = [col+'_x' if i % 2 == 0 else col+'_y' for i, col in enumerate(cols)]
        cols = np.concatenate([df.columns[:3], cols])
        df.columns = cols

    df_ball = df_away[['Period', 'Frame', 'Time [s]', 'Ball_x', 'Ball_y']].copy()
    df_home.drop(['Ball_x', 'Ball_y'], axis=1, inplace=True)
    df_away.drop(['Ball_x', 'Ball_y'], axis=1, inplace=True)
    df_ball.rename({'Ball_x': 'x', 'Ball_y': 'y'}, axis=1, inplace=True)


    #Detect attacking direction
    first_home_frame = df_home.loc[0, [c for c in df_home.columns if 'x' in c]]
    first_away_frame = df_away.loc[0, [c for c in df_away.columns if 'x' in c]]

    home_x_mean = first_home_frame.mean()
    away_x_mean = first_away_frame.mean()

    if home_x_mean < away_x_mean:
        directions = {
            home_team: {1: 1, 2:-1},
            away_team: {1: -1, 2: 1}
        }
    else:
        directions = {
            home_team: {1: -1, 2: 1},
            away_team: {1: 1, 2: -1}
        }
    
        
    return df_home, df_away, df_ball, directions

def add_direction(df, direction_map):
    df = df.copy()
    df['Attacking_Direction'] = df.apply(
        lambda row: direction_map.get(row['Team'], {}).get(row['Period']), 
        axis=1
    )
    return df

def invert_y_axis(df):
    df = df.copy()
    df['Start Y'] = 1 - df['Start Y']
    df['End Y'] = 1 - df['End Y']
    return df

def invert_y_axis_tracking(df, ball=False):
    df = df.copy()
    if ball:
        y_cols = ['y']
    else:
        y_cols = [c for c in df.columns if c.endswith('_y')]
    df[y_cols] = 1 - df[y_cols]
    return df

def to_coordinates(df, pitch_length=105, pitch_width=68):
    df = df.copy()
    df['Start X'] = df['Start X'] * pitch_length
    df['End X'] = df['End X'] * pitch_length
    df['Start Y'] = df['Start Y'] * pitch_width
    df['End Y'] = df['End Y'] * pitch_width
    return df

def to_coordinates_tracking(df, pitch_length=105, pitch_width=68, ball=False):
    df = df.copy()
    if ball:
        x_cols = ['x']
        y_cols = ['y']
    else:
        x_cols = [c for c in df.columns if c.endswith('_x')]
        y_cols = [c for c in df.columns if c.endswith('_y')]
    df[x_cols] = df[x_cols] * pitch_length
    df[y_cols] = df[y_cols] * pitch_width
    return df

def get_event_zone(event, pitch_length=1, pitch_width=1):
    import pandas as pd
    attacking_direction = event['Attacking_Direction']
    x_coord_start = event['Start X']
    y_coord_start = event['Start Y']
    x_zone_start = ''
    y_zone_start = ''

    # --- Normalize and check for 'Unknown' or pd.NA/nan on start coordinates ---
    is_x_start_unknown = (isinstance(x_coord_start, str) and x_coord_start == 'Unknown') or pd.isna(x_coord_start)
    is_y_start_unknown = (isinstance(y_coord_start, str) and y_coord_start == 'Unknown') or pd.isna(y_coord_start)
    try:
        x_coord_start = float(x_coord_start)
        y_coord_start = float(y_coord_start)
    except (ValueError, TypeError):
        is_x_start_unknown = True
        is_y_start_unknown = True

    if is_x_start_unknown or is_y_start_unknown:
        x_zone_start = 'Unknown'
        y_zone_start = 'Unknown'
    else:
        if x_coord_start <= 1/3 * pitch_length:
            x_zone_start = 'Tercio Defensivo' if attacking_direction == 1 else 'Tercio Ofensivo'
        elif x_coord_start <= 2/3 * pitch_length:
            x_zone_start = 'Tercio Medio'
        else:
            x_zone_start = 'Tercio Ofensivo' if attacking_direction == 1 else 'Tercio Defensivo'

        if attacking_direction == 1:
            if y_coord_start <= 1/5 * pitch_width:
                y_zone_start = 'Banda Izquierda'
            elif y_coord_start <= 2/5 * pitch_width:
                y_zone_start = 'Carril Izquierdo'
            elif y_coord_start <= 3/5 * pitch_width:
                y_zone_start = 'Centro'
            elif y_coord_start <= 4/5 * pitch_width:
                y_zone_start = 'Carril Derecho'
            else:
                y_zone_start = 'Banda Derecha'
        else:
            if y_coord_start <= 1/5 * pitch_width:
                y_zone_start = 'Banda Derecha'
            elif y_coord_start <= 2/5 * pitch_width:
                y_zone_start = 'Carril Derecho'
            elif y_coord_start <= 3/5 * pitch_width:
                y_zone_start = 'Centro'
            elif y_coord_start <= 4/5 * pitch_width:
                y_zone_start = 'Carril Izquierdo'
            else:
                y_zone_start = 'Banda Izquierda'

    # --- Normalize and check for 'Unknown' or pd.NA/nan on end coordinates ---
    x_coord_end = event.get('End X', None)
    y_coord_end = event.get('End Y', None)
    x_zone_end = None
    y_zone_end = None

    is_x_end_unknown = (isinstance(x_coord_end, str) and x_coord_end == 'Unknown') or pd.isna(x_coord_end)
    is_y_end_unknown = (isinstance(y_coord_end, str) and y_coord_end == 'Unknown') or pd.isna(y_coord_end)
    try:
        x_coord_end = float(x_coord_end)
        y_coord_end = float(y_coord_end)
    except (ValueError, TypeError):
        is_x_end_unknown = True
        is_y_end_unknown = True

    if is_x_end_unknown and is_y_end_unknown:
        x_zone_end = 'Unknown'
        y_zone_end = 'Unknown'
    else:
        if not is_x_end_unknown:
            if x_coord_end <= 1/3 * pitch_length:
                x_zone_end = 'Tercio Defensivo' if attacking_direction == 1 else 'Tercio Ofensivo'
            elif x_coord_end <= 2/3 * pitch_length:
                x_zone_end = 'Tercio Medio'
            else:
                x_zone_end = 'Tercio Ofensivo' if attacking_direction == 1 else 'Tercio Defensivo'
        else:
            x_zone_end = 'Unknown'

        if not is_y_end_unknown:
            if attacking_direction == 1:
                if y_coord_end <= 1/5 * pitch_width:
                    y_zone_end = 'Banda Izquierda'
                elif y_coord_end <= 2/5 * pitch_width:
                    y_zone_end = 'Carril Izquierdo'
                elif y_coord_end <= 3/5 * pitch_width:
                    y_zone_end = 'Centro'
                elif y_coord_end <= 4/5 * pitch_width:
                    y_zone_end = 'Carril Derecho'
                else:
                    y_zone_end = 'Banda Derecha'
            else:
                if y_coord_end <= 1/5 * pitch_width:
                    y_zone_end = 'Banda Derecha'
                elif y_coord_end <= 2/5 * pitch_width:
                    y_zone_end = 'Carril Derecho'
                elif y_coord_end <= 3/5 * pitch_width:
                    y_zone_end = 'Centro'
                elif y_coord_end <= 4/5 * pitch_width:
                    y_zone_end = 'Carril Izquierdo'
                else:
                    y_zone_end = 'Banda Izquierda'
        else:
            y_zone_end = 'Unknown'

    return (x_zone_start, y_zone_start, x_zone_end, y_zone_end)

def to_long_form(df):
    """ Pivots a dataframe from wide-form (each player as a separate column) to long form (rows)"""
    df = pd.melt(df, id_vars=df.columns[:3], value_vars=df.columns[3:], var_name='player')
    df.loc[df.player.str.contains('_x'), 'coordinate'] = 'x'
    df.loc[df.player.str.contains('_y'), 'coordinate'] = 'y'
    df = df.dropna(axis=0, how='any')
    df['player'] = df.player.str[6:-2]
    df = (df.set_index(['Period', 'Frame', 'Time [s]', 'player', 'coordinate'])['value']
          .unstack()
          .reset_index()
          .rename_axis(None, axis=1))
    return df


### Parte II: Generación de Playback de Jugada

In [3]:
def plot_tracking_animation(
    df_home, 
    df_away, 
    df_ball, 
    start_frame, 
    end_frame, 
    show_names=False, 
    home_team_name="Home", 
    away_team_name="Away", 
    home_color="#7f63b8", 
    away_color="#b94b75",
    save_path=None
):
    """
    Plots an animated match sequence using mplsoccer Pitch,
    showing home and away players and the ball.

    Args:
        df_home (pd.DataFrame): DataFrame with home player positions (must have cols 'Frame', 'x', 'y')
        df_away (pd.DataFrame): DataFrame with away player positions (must have cols 'Frame', 'x', 'y')
        df_ball (pd.DataFrame): DataFrame with ball positions (must have cols [3]='x', [4]='y', [1]='Frame')
        show_names (bool): Whether to show player IDs/names on the pitch (default: False)
        home_team_name (str): Name of the home team, default "Home"
        away_team_name (str): Name of the away team, default "Away"
        home_color (str): Color for home team, default "#7f63b8"
        away_color (str): Color for away team, default "#b94b75"
    Returns:
        anim: The matplotlib.animation.FuncAnimation object.
    """
    df_home = df_home.loc[(df_home.Frame >= start_frame) & (df_home.Frame <= end_frame)].copy()
    df_away = df_away.loc[(df_away.Frame >= start_frame) & (df_away.Frame <= end_frame)].copy()    
    df_ball = df_ball.loc[(df_ball.Frame >= start_frame) & (df_ball.Frame <= end_frame)].copy()

    df_home_long = to_long_form(df_home)
    df_away_long = to_long_form(df_away)

    # Get unique players
    home_players = df_home_long['player'].unique()
    away_players = df_away_long['player'].unique()

    pitch = Pitch(pitch_type='uefa', goal_type='line', )
    fig, ax = pitch.draw(figsize=(16, 10.4))

    # Animated objects
    marker_kwargs = {'marker': 'o', 'markeredgecolor': 'black', 'linestyle': 'None'}
    ball_plot, = ax.plot([], [], ms=6, markerfacecolor='w', zorder=3, **marker_kwargs)
    away_plot, = ax.plot([], [], ms=10, markerfacecolor=away_color, **marker_kwargs)
    home_plot, = ax.plot([], [], ms=10, markerfacecolor=home_color, **marker_kwargs)

    # Create text objects upfront (one per player) - store as lists for easier management
    home_texts = []
    away_texts = []
    
    if show_names:
        for player in home_players:
            txt = ax.text(
                0, 0, str(player), fontsize=10,
                color=home_color, ha='center', va='bottom', fontweight='bold', zorder=10,
                bbox=dict(facecolor='white', alpha=0.8, edgecolor=home_color, linewidth=1, boxstyle='round,pad=0.2'),
                visible=False, clip_on=False
            )
            home_texts.append(txt)
        for player in away_players:
            txt = ax.text(
                0, 0, str(player), fontsize=10,
                color=away_color, ha='center', va='bottom', fontweight='bold', zorder=10,
                bbox=dict(facecolor='white', alpha=0.8, edgecolor=away_color, linewidth=1, boxstyle='round,pad=0.2'),
                visible=False, clip_on=False
            )
            away_texts.append(txt)

    # Create colored title
    fig_text(
        0.515, 0.97, f"<{home_team_name}> vs <{away_team_name}>", size=25, fig=fig,
        highlight_textprops=[{"color": home_color}, {"color": away_color}],
        ha="center",  color="black"
    )

    def animate(i):
        frame = df_ball.iloc[i]['Frame']
        # Ball
        ball_plot.set_data([df_ball.iloc[i]['x']], [df_ball.iloc[i]['y']])
        
        # Away players
        away_frame = df_away_long.loc[df_away_long.Frame == frame]
        away_x = away_frame['x'].values
        away_y = away_frame['y'].values
        away_plot.set_data(away_x, away_y)
        
        # Home players
        home_frame = df_home_long.loc[df_home_long.Frame == frame]
        home_x = home_frame['x'].values
        home_y = home_frame['y'].values
        home_plot.set_data(home_x, home_y)
        
        # Update text labels
        if show_names:
            # Hide all texts first
            for txt in home_texts + away_texts:
                txt.set_visible(False)
            
            # Update home player labels
            for idx, (_, row) in enumerate(home_frame.iterrows()):
                if idx < len(home_texts):
                    home_texts[idx].set_position((row['x'], row['y'] + 1.5))
                    home_texts[idx].set_text(str(row['player']))
                    home_texts[idx].set_visible(True)
            
            # Update away player labels
            for idx, (_, row) in enumerate(away_frame.iterrows()):
                if idx < len(away_texts):
                    away_texts[idx].set_position((row['x'], row['y'] + 1.5))
                    away_texts[idx].set_text(str(row['player']))
                    away_texts[idx].set_visible(True)
            
            # Return all artists but disable blit for text (text doesn't blit well)
            return [ball_plot, away_plot, home_plot] + home_texts + away_texts
        else:
            return ball_plot, away_plot, home_plot

    # Use blit=False if showing names, as text doesn't blit well
    anim = animation.FuncAnimation(
        fig, animate, frames=len(df_ball), interval=50, blit=not show_names
    )

    if save_path:
        writer = "pillow" if save_path.lower().endswith(".gif") else "ffmpeg"
        anim.save(save_path, writer=writer, dpi=100)

    # Display animation in Jupyter
    plt.close(fig)
    return HTML(anim.to_jshtml())

### Parte III: Procesamiento con LVM

In [4]:
def extract_frames(video_path, interval=30):
    """
    Extracts frames from a video file at a specified interval.
    Returns a list of base64 encoded strings.
    """
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Video file not found: {video_path}")

    video = cv2.VideoCapture(video_path)
    base64Instances = []
    count = 0

    while video.isOpened():
        success, frame = video.read()
        if not success:
            break

        if count % interval == 0:
            # Resize to reduce token usage (optional, e.g., max 1024px)
            # frame = cv2.resize(frame, (1024, int(1024 * frame.shape[0] / frame.shape[1])))
            
            _, buffer = cv2.imencode(".jpg", frame)
            base64Instances.append(base64.b64encode(buffer).decode("utf-8"))

        count += 1

    video.release()
    print(f"🎥 Extracted {len(base64Instances)} frames from {video_path}")
    return base64Instances

TODO: Prompt engineering para mejorar instrucciones, incluir contexto de los equipos (saber alineaciones, posiciones, y dorsales de jugadores para nombrarlos directamente), etc. Para pasar contexto, podríamos usar jinja2 para construir un template que se rellene con DTOs de equipos, jugadores, etc.

In [20]:
def describe_video_local(frames, prompt:str, model='llava'):
    """
    Sends frames to the local VLM (Ollama) and gets a description.
    
    Args:
        frames (list): List of base64 encoded strings or byte sequences.
        model (str): The Ollama model to use (default: 'llava').
        target_frames (int): Number of frames to sample and analyze (default: 10).
    """
    
    print(f"🧠 Sending request to Ollama (model: {model})...")
    response = ollama.chat(
        model=model,
        messages=[{
            'role': 'user',
            'content': prompt,
            'images': frames
        }]
    )

    description = response['message']['content'].strip()
    return description


### Parte IV: Text-to-Speech

In [6]:
async def generate_audio(text, output_file="commentary.mp3", voice="es-ES-AlvaroNeural"):
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(output_file)
    return output_file

def play_audio(file_path):
    return Audio(file_path, autoplay=False)

## Pipeline: End-to-End Execution

### Parte I: Extracción de Fases de Juego

In [7]:
matplotlib.rcParams['animation.embed_limit'] = 100.0

data_folder = osp.join( '..', '..', 'data', 'metrica')

MATCH_ID = 1
home_team = 'Home'
away_team = 'Away'
match_folder = osp.join(data_folder, f'Sample_Game_{MATCH_ID}')
TRACKING_HOME_FILE = osp.join(match_folder, f'Sample_Game_{MATCH_ID}_RawTrackingData_{home_team}_Team.csv')
TRACKING_AWAY_FILE = osp.join(match_folder, f'Sample_Game_{MATCH_ID}_RawTrackingData_{away_team}_Team.csv')
EVENTS_FILE = osp.join(match_folder, f'Sample_Game_{MATCH_ID}_RawEventsData.csv')

In [8]:
# Load Tracking Data
tracking_home, tracking_away, tracking_ball, directions = load_tracking_as_mplsoccer(
    TRACKING_HOME_FILE,
    TRACKING_AWAY_FILE
)

tracking_home = invert_y_axis_tracking(tracking_home)
tracking_away = invert_y_axis_tracking(tracking_away)
tracking_ball = invert_y_axis_tracking(tracking_ball, ball=True)

tracking_home = to_coordinates_tracking(tracking_home)
tracking_away = to_coordinates_tracking(tracking_away)
tracking_ball = to_coordinates_tracking(tracking_ball, ball=True)

In [9]:
# Process Events and Phases
events = pd.read_csv(EVENTS_FILE)
events = add_direction(events, directions)    
events = invert_y_axis(events)
zone_cols = events.apply(get_event_zone, axis=1, result_type='expand')
zone_cols.columns = ['Start_third', 'Start_channel', 'End_third', 'End_channel']
events = events.join(zone_cols)

phases_df = build_pro_tactical_phases(events)
phases_df.head()

,Phase_ID,Team,Period,Start_Time,Duration,Event_Count,Start_Frame,End_Frame,Start_X,End_X,Events_List,Subtypes_List,Events_Ids,Start_Type,Start_Subtype,Start_Zone,End_Zone,Context,Outcome
0,1,Away,1,0.04,19.88,15,1,498,0.45,0.33,"[SET PIECE, PASS, PASS, PASS, PASS, PASS, PASS...","[KICK OFF, nan, nan, nan, nan, nan, nan, INTER...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",SET PIECE,KICK OFF,Creación,Finalización,ABP,Pérdida
1,2,Home,1,19.92,4.80,4,498,618,0.36,0.67,"[PASS, PASS, BALL LOST, RECOVERY]","[HEAD, nan, INTERCEPTION, INTERCEPTION]","[15, 16, 17, 18]",PASS,HEAD,Creación,Finalización,Recuperación,Pérdida
2,3,Away,1,30.52,14.84,6,763,1134,0.58,0.31,"[PASS, PASS, PASS, PASS, BALL LOST, RECOVERY]","[nan, nan, nan, nan, INTERCEPTION, INTERCEPTION]","[19, 20, 21, 22, 23, 24]",PASS,INTERCEPTION,Creación,Finalización,Recuperación,Pérdida
3,4,Home,1,45.36,9.60,6,1134,1425,0.32,1.05,"[PASS, PASS, PASS, BALL LOST, RECOVERY, BALL OUT]","[HEAD, nan, nan, INTERCEPTION, INTERCEPTION, nan]","[25, 26, 27, 28, 29, 30]",PASS,HEAD,Iniciación,Finalización,Recuperación,Pérdida
4,5,Home,1,85.72,5.84,4,2143,2309,1.00,1.01,"[SET PIECE, PASS, PASS, SHOT]","[CORNER KICK, nan, CROSS, HEAD-ON TARGET-GOAL]","[31, 32, 33, 34]",SET PIECE,CORNER KICK,Finalización,Finalización,ABP,GOL


In [10]:
phases_df.shape[0]

188

### Parte II: Generación de Playback de la Jugada

In [11]:
PHASE_ID = 9 # Change this ID to select a different phase
# ==================================\n

print(f"🎬 Processing Phase ID: {PHASE_ID}")

# Get Phase Details
phase_row = phases_df.loc[phases_df['Phase_ID'] == PHASE_ID]
if phase_row.empty:
    raise ValueError(f"Phase ID {PHASE_ID} not found in phases_df")

start_frame = phase_row.iloc[0]['Start_Frame']
end_frame = phase_row.iloc[0]['End_Frame']
team_name = phase_row.iloc[0]['Team']

print(f"⏱️ Frame Range: {start_frame} - {end_frame} ({end_frame - start_frame} frames)")
print(f"⚽ Team: {team_name}")

VIDEO_FILENAME = f"phase_{PHASE_ID}.mp4"

print(f"🎥 Generating video: {VIDEO_FILENAME}...")
# Generate and Save Video
anim = plot_tracking_animation(
    tracking_home, 
    tracking_away, 
    tracking_ball, 
    start_frame=start_frame, 
    end_frame=end_frame, 
    show_names=True,
    save_path=VIDEO_FILENAME
)

print(f"✅ Video saved as: {VIDEO_FILENAME}")

🎬 Processing Phase ID: 9
⏱️ Frame Range: 5149 - 5385 (236 frames)
⚽ Team: Home
🎥 Generating video: phase_9.mp4...
✅ Video saved as: phase_9.mp4


### Parte III: Procesamiento con LVM

In [12]:
phases_df.iloc[phases_df['Phase_ID'] == PHASE_ID]

,Phase_ID,Team,Period,Start_Time,Duration,Event_Count,Start_Frame,End_Frame,Start_X,End_X,Events_List,Subtypes_List,Events_Ids,Start_Type,Start_Subtype,Start_Zone,End_Zone,Context,Outcome
8,9,Home,1,205.96,9.44,9,5149,5385,0.24,0.64,"[PASS, BALL LOST, CHALLENGE, CHALLENGE, RECOVE...","[nan, INTERCEPTION, GROUND-LOST, GROUND-WON, I...","[57, 58, 59, 60, 61, 62, 63, 64, 65]",PASS,INTERCEPTION,Iniciación,Creación,Recuperación,Pérdida


In [13]:
from pydantic import BaseModel
from typing import List, Dict, Any

class PlayerDTO(BaseModel):
    id: str
    name: str
    position: str
    team: str

class PhaseDTO(BaseModel):
    home_team: str
    away_team: str
    mm_ss: str
    duration: float
    team_in_posession: str
    start_zone: str
    end_zone: str
    outcome: str
    players: List[PlayerDTO]

players_dtos = [
    # --- Home  ---
    PlayerDTO(id="Player1",  name="Jules Kounde",     position="",  team="Home"),
    PlayerDTO(id="Player2",  name="Pau Cubarsí",      position="",  team="Home"),
    PlayerDTO(id="Player3",  name="Eric García",      position="",  team="Home"),
    PlayerDTO(id="Player4",  name="Alejandro Balde",  position="",  team="Home"),
    PlayerDTO(id="Player5",  name="Fermín López",     position="",  team="Home"),
    PlayerDTO(id="Player6",  name="Frenkie de Jong",  position="",  team="Home"),
    PlayerDTO(id="Player7",  name="Dani Olmo",        position="",  team="Home"),
    PlayerDTO(id="Player8",  name="Raphinha",         position="",  team="Home"),
    PlayerDTO(id="Player9",  name="Lewandowski",      position="",  team="Home"),
    PlayerDTO(id="Player10", name="Lamine Yamal",     position="",  team="Home"),
    PlayerDTO(id="Player11", name="Joan Garcia",      position="",  team="Home"),
    # --- Away
    PlayerDTO(id="Player15", name="Yann Aurel Bisseck",  position="",  team="Away"),
    PlayerDTO(id="Player16", name="Alessandro Bastoni",  position="",  team="Away"),
    PlayerDTO(id="Player17", name="Manuel Akanji",       position="",  team="Away"),
    PlayerDTO(id="Player18", name="Henrikh Mkhitaryan",  position="",  team="Away"),
    PlayerDTO(id="Player19", name="Luis Henrique",       position="",  team="Away"),
    PlayerDTO(id="Player20", name="Petar Sucic",         position="",  team="Away"),
    PlayerDTO(id="Player21", name="Piotr Zielinski",     position="",  team="Away"),
    PlayerDTO(id="Player22", name="Federico Dimarco",    position="",  team="Away"),
    PlayerDTO(id="Player23", name="Lautaro Martínez",    position="",  team="Away"),
    PlayerDTO(id="Player24", name="Marcus Thuram",       position="",  team="Away"),
    PlayerDTO(id="Player25", name="Yann Sommer",         position="",  team="Away")
]
    
phase_row = phases_df.iloc[phases_df['Phase_ID'] == PHASE_ID]
phase_dto = PhaseDTO(
    home_team="FC Barcelona",
    away_team="FC Internazionale Milano",
    mm_ss=f"{int(phase_row['Start_Time'].iloc[0] // 60):02d}:{int(phase_row['Start_Time'].iloc[0] % 60):02d}",
    duration=float(phase_row['Duration'].iloc[0]),
    team_in_posession=phase_row['Team'].iloc[0],
    start_zone=phase_row['Start_Zone'].iloc[0],
    end_zone=phase_row['End_Zone'].iloc[0],
    outcome=phase_row['Outcome'].iloc[0],
    players=players_dtos
)

from jinja2 import Environment, FileSystemLoader

# Initialize Jinja2 environment and load the template
env = Environment(loader=FileSystemLoader("../../prompts/video_to_speech"))
template = env.get_template("context.j2")

# Render the template with the DTO data
rendered_context = template.render(phase_dto.model_dump())


In [14]:
print(f"🚀 Starting processing for: {VIDEO_FILENAME}")

frames = extract_frames(VIDEO_FILENAME, interval=5)
with open('../../prompts/video_to_speech/instructions.md', 'r', encoding='utf-8') as f:
    instructions = f.read()

prompt = instructions + "\n\n" + rendered_context
print(prompt)

🚀 Starting processing for: phase_9.mp4
🎥 Extracted 48 frames from phase_9.mp4
# Perfil e instrucciones básicas

Eres un **Observador Táctico de Alta Precisión**. Tu única función es **registrar objetivamente** todo lo que ocurre en los clips de video. NO debes interpretar intenciones y NO debes juzgar si una jugada es buena o mala. Tu trabajo es generar un **registro detallado de hechos observables** (data logging) que servirá como base de datos para futuras consultas.

> Importante: Empieza directamente con la descripción de la jugada, sin preámbulos ni introducciones. 

## Objetivo
Generar una descripción densa y granular de los eventos, posiciones y movimientos. Cada frase debe contener datos fácticos recuperables mediante búsqueda (RAG).

## Marco de Observación (Vocabulario Controlado)
Usa estrictamente esta terminología para describir ubicaciones y acciones:

### 1. Ubicación Espacial (Factores Observables)
- **Zonas Verticales**: Zona 1 (Inicio), Zona 2 (Creación), Zona 3 (Final

In [23]:
description = describe_video_local(frames, prompt=prompt) #, model="qwen3-vl:4b")
print("\n📝 Generated Description:")
print(description)

🧠 Sending request to Ollama (model: llava)...

📝 Generated Description:
FC Barcelona vs FC Internazionale Milano (Away Team)
Tiempo de Inicio: 03:25
Duración: 9.43999999 segundos
Resultado: Pérdida
Posición del Equipo: 1º

Inicio (Zona 1/Carril Izquierdo)
- **Jules Koundé** (ID: Player1) se encuentra en la defensa, posicionándose atrás de los demás jugadores.
- **Pau Cubarte** (ID: Player2) y **Eric García** (ID: Player3) son dos jugadores en línea detrás del equipo rival, mientras que **Alejandro Balde** (ID: Player4) está más avanzado hacia la defensa central.
- Los jugadores de FC Barcelona se mantienen en su propia zona defensiva y no hay contacto con el balón al inicio.

Creación (Zona 2/Carril Central)
- Después de la pérdida, **Frenkie de Jong** (ID: Player6) entra en juego desde su posición central y se dirige hacia la defensa rival.
- Mientras tanto, los jugadores del equipo visitante están más adelantados que los jugadores del equipo local, con **Lautaro Martínez** (ID: Playe

### Parte IV: Text-to-Speech

In [17]:
print("\n🗣️ Synthesizing Speech...")
audio_file = await generate_audio(description)
print("✅ Done!")

play_audio(audio_file)


🗣️ Synthesizing Speech...
✅ Done!
